# 03 – RQ2: pose features against raw appearance
In this notebook, we run the full-scale version of
`scripts/train_rq2_local.py`. We train on several Okutama videos and hold
out whole videos rather than single tracks. We then compare a classifier
that uses pose features (the PoseMLP) against one that uses raw image crops
(the AppearanceCNN). Both are trained with the same procedure, so the
comparison is fair.


In [ ]:
# Install the packages we need.
!pip -q install ultralytics rtmlib onnxruntime-gpu
import torch
print('CUDA available:', torch.cuda.is_available())

# Load the project code.
from pathlib import Path
SRC_ZIP = None
if SRC_ZIP is None:
    from google.colab import files
    up = files.upload()
    SRC_ZIP = next(iter(up))
!mkdir -p /content/project && unzip -q -o "$SRC_ZIP" -d /content/project
import sys
sys.path.insert(0, '/content/project/src')
sys.path.insert(0, '/content/project')

# Results are saved to Google Drive so they survive a disconnect.
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/sar_project_results'); OUT.mkdir(parents=True, exist_ok=True)


CUDA available: True


Saving src.zip to src (1).zip
Mounted at /content/drive


In [ ]:
# Download Okutama-Action files from the public Dropbox folder.
OKUTAMA_BASE = ('https://www.dropbox.com/scl/fo/9qvpsb3fsamvqzsa12149/'
                'APTyV-f01XLnJ0WFpZSBLOE?preview={name}&rlkey=7u7131amaul29amyr4jbnnu03&dl=1')

def fetch_okutama(name, dest='/content/data/okutama'):
    """Download and unpack one Okutama archive unless it is already present."""
    import subprocess, pathlib
    d = pathlib.Path(dest); d.mkdir(parents=True, exist_ok=True)
    zp = d / name
    if not zp.exists():
        subprocess.run(['curl', '-L', '-o', str(zp), OKUTAMA_BASE.format(name=name)], check=True)
    subprocess.run(['unzip', '-q', '-o', str(zp), '-d', str(d)], check=True)
    return d


In [ ]:
# Download the videos. Start with the sample; the full set is about 5 GB.
fetch_okutama('Sample.zip')
fetch_okutama('TrainSetVideos.zip')
fetch_okutama('TestSetVideos.zip')
import glob
videos = sorted(glob.glob('/content/data/okutama/**/*.mov', recursive=True))
labels = sorted(glob.glob('/content/data/okutama/**/*.txt', recursive=True))
print(len(videos), 'videos')


4 videos


In [ ]:
import numpy as np
import config
from pathlib import Path
config.MODELS_DIR = OUT; config.TABLES_DIR = OUT; config.FIGURES_DIR = OUT
from scripts.train_rq2_local import extract, build_dataset
from data.okutama import parse_annotations

all_tracks, all_crops, offset = {}, {}, 0
for v, t in zip(videos, labels):
    frames = parse_annotations(t)
    tr, cr = extract(v, frames, step=2, pose_device='cuda')
    # Shift the track ids so videos do not collide; the offset also encodes the video.
    for tid, seq in tr.items():
        all_tracks[offset + tid] = seq
    for (tid, f), c in cr.items():
        all_crops[(offset + tid, f)] = c
    offset += 10_000
X, C, y, tids, classes = build_dataset(all_tracks, all_crops)


Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.zip" to /root/.cache/rtmlib/hub/checkpoints/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.zip
100%|██████████| 48.4M/48.4M [00:01<00:00, 26.4MB/s]
/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


load /root/.cache/rtmlib/hub/checkpoints/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.onnx with onnxruntime backend
[extract] Pose backend: rtmpose
  Processed 100 labeled frames (video frame 198)
  Processed 200 labeled frames (video frame 398)
  Processed 300 labeled frames (video frame 598)
  Processed 400 labeled frames (video frame 798)
  Processed 500 labeled frames (video frame 998)
  Processed 600 labeled frames (video frame 1198)
  Processed 700 labeled frames (video frame 1398)
  Processed 800 labeled frames (video frame 1598)
  Processed 900 labeled frames (video frame 1798)
  Processed 1000 labeled frames (video frame 1998)
  Processed 1100 labeled frames (video frame 2198)
[extract] 1125 frames, 99 tracks, 7615 crops
load /root/.cache/rtmlib/hub/checkpoints/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.onnx with onnxruntime backend
[extract] Pose backend: rtmpose
  Processed 100 labeled frames (video frame 198)
  Processed 200 labeled frame

In [ ]:
# Split by video (tids // 10_000), which is stricter than splitting by track.
vid_of = tids // 10_000
val_vids = set(list(sorted(set(vid_of)))[-max(1, len(set(vid_of))//4):])
va = np.isin(vid_of, list(val_vids)); tr = ~va
print('Train', tr.sum(), 'val', va.sum())

from classify import PoseMLP, AppearanceCNN, train_classifier, predict, evaluate
counts = np.bincount(y[tr], minlength=len(classes)).astype('float32')
w = counts.sum()/np.maximum(counts,1)/len(classes)

# Both models use the same training function, split and class weights.
pose_model,_ = train_classifier(PoseMLP(len(classes)), X[tr], y[tr], X[va], y[va],
                                epochs=60, class_weights=w, device='cuda')
m_pose = evaluate(y[va], predict(pose_model, X[va], device='cuda'), classes, 'pose_mlp_full')

Cn = (C.astype('float32')/255.).transpose(0,3,1,2)
cnn,_ = train_classifier(AppearanceCNN(len(classes)), Cn[tr], y[tr], Cn[va], y[va],
                         epochs=60, class_weights=w, device='cuda')
m_app = evaluate(y[va], predict(cnn, Cn[va], device='cuda'), classes, 'appearance_cnn_full')

import pandas as pd
pd.DataFrame([{'model':'PoseMLP', **m_pose['per_class_f1'], 'macro_f1':m_pose['macro_f1']},
              {'model':'AppearanceCNN', **m_app['per_class_f1'], 'macro_f1':m_app['macro_f1']}]
             ).to_csv(OUT/'rq2_full.csv', index=False)


Train 3366 val 199
  epoch   0 loss 1.1872 val macro-F1 0.230
  epoch   5 loss 0.8141 val macro-F1 0.356
  epoch  10 loss 0.6878 val macro-F1 0.371
  epoch  15 loss 0.5966 val macro-F1 0.346
              precision    recall  f1-score   support

      mobile       0.62      0.76      0.68       105
  motionless       0.00      0.00      0.00         0
  stationary       0.65      0.38      0.48        94

    accuracy                           0.58       199
   macro avg       0.42      0.38      0.39       199
weighted avg       0.63      0.58      0.59       199

Saved /content/drive/MyDrive/sar_project_results/confusion_pose_mlp_full.png
  epoch   0 loss 0.7795 val macro-F1 0.289
  epoch   5 loss 0.5410 val macro-F1 0.267
              precision    recall  f1-score   support

      mobile       0.56      0.58      0.57       105
  motionless       0.00      0.00      0.00         0
  stationary       0.48      0.43      0.45        94

    accuracy                           0.51    

For the ablations, we rerun the two cells above with one change at a time:
the window size (5, 15 or 30), the crop resolution (32, 64 or 96), and the
pose features with and without the temporal part (slice `X[:, :41]`). We
also save about 20 misclassified crops and inspect them for the error
analysis.
